# Lab Tracking trên Colab — 5 video, chọn tracker cho từng cảnh

**Trước khi chạy:** `Runtime → Change runtime type → T4 GPU`. Sau đó chạy lần lượt từ trên xuống.

Notebook làm đúng các bước trong `HUONG_DAN.md`, chỉ khác là chạy trên Colab:

| Phần | Việc | Bước trong HUONG_DAN |
|---|---|---|
| 1 | Clone repo + TrackEval, cài thư viện | C |
| 2 | Tải ảnh 5 video, `check_data.py` | C |
| 3 | Hàm tiện ích (chạy tracker, chấm, xem video) | — |
| 4 | Baseline: `video_1` + ByteTrack, 150 frame | D |
| 5 | `video_1` có nhãn: so 5 tracker bằng HOTA / MOTA / IDF1, rồi quét `conf` / `iou` | E + F |
| 6 | `video_2`–`video_5`: so tracker chuyển động với tracker Re-ID **bằng mắt**, quét `conf` / `iou` | E |
| 7 | Chạy **đủ frame** cấu hình đã chọn vào `runs/nop_bai/` | E.3 |
| 8 | Chấm `video_1` bản nộp, lấy bảng dán vào báo cáo | F |
| 9 | Nén và tải kết quả về máy / lưu Drive | Nộp bài |

**Luật chơi (không đổi):** detector `yolo26n.pt`, ảnh 640 px, lớp người, Re-ID `osnet_x0_25_msmt17.pt`.
**Bạn chọn:** tracker (`bytetrack`, `ocsort`, `botsort`, `strongsort`, `deepocsort`), `conf`, `iou` của detector.

Colab xóa file khi ngắt runtime. Làm xong phần 7–8 thì chạy ngay phần 9.

## 1. Cài đặt

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Le-Anh-Duy/K4-Track4-2A202602723-2A202602437-Day22-Tracking-Student.git"
REPO = Path("/content/K4-Track4-2A202602723-2A202602437-Day22-Tracking-Student")

if not REPO.exists():
    !git clone -q {REPO_URL} {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
if not Path("TrackEval").exists():
    !git clone -q https://github.com/JonathonLuiten/TrackEval.git

# boxmot 10.0.42 khóa numpy==1.23.1 (không có bản cho Python của Colab) nên cài không kèm phụ thuộc,
# rồi tự cài các gói nó thật sự dùng. setuptools<81 vì boxmot còn import pkg_resources.
!pip install -q "setuptools<81" "ultralytics>=8.4" "lap>=0.4.0" filterpy gdown loguru yacs ftfy
!pip install -q --no-deps boxmot==10.0.42
!pip install -q -e TrackEval/

# boxmot 10.0.42 gọi float() trên mảng 1 phần tử; NumPy >= 2.4 coi đó là lỗi
# (DeepOCSORT / OCSORT / StrongSORT dừng giữa chừng). Vá tại chỗ — chạy lại ô này không sao.
import importlib.util

BOXMOT_DIR = Path(importlib.util.find_spec("boxmot").submodule_search_locations[0])
VA_BOXMOT = {
    "motion/kalman_filters/kalman_filter.py": [
        ("r = w / float(h)", "r = w / np.asarray(h).item()"),
        ("sqrt(float(dot(dot(y.T, self.SI), y)))", "sqrt(np.asarray(dot(dot(y.T, self.SI), y)).item())"),
        ("sqrt(float(dot(dot(self.y.T, self.SI), self.y)))",
         "sqrt(np.asarray(dot(dot(self.y.T, self.SI), self.y)).item())"),
    ],
    "motion/kalman_filters/adapters/strongsort_kf_adapter.py": [
        ("np.sqrt(float(np.dot(np.dot(y.T, self.SI[:2, :2]), y)))",
         "np.sqrt(np.asarray(np.dot(np.dot(y.T, self.SI[:2, :2]), y)).item())"),
    ],
}
for rel, pairs in VA_BOXMOT.items():
    path = BOXMOT_DIR / rel
    src = path.read_text()
    for old, new in pairs:
        src = src.replace(old, new)
    path.write_text(src)
print("Đã vá boxmot:", BOXMOT_DIR)

In [ ]:
import numpy as np
import torch
import ultralytics
import boxmot

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"numpy {np.__version__} | torch {torch.__version__} | ultralytics {ultralytics.__version__} | boxmot {boxmot.__version__}")
if DEVICE == "cpu":
    print("CHƯA có GPU — Runtime → Change runtime type → T4 GPU, rồi chạy lại từ đầu.")
else:
    print("GPU:", torch.cuda.get_device_name(0))
!python -m pytest -q

## 2. Dữ liệu

Tải `data_lab21.zip` từ Google Drive của giảng viên rồi giải nén vào `/content/lab_data`.
Nếu `gdown` báo quá lượt tải, tải file về máy rồi upload vào `/content/data_lab21.zip`, chạy lại ô.

Gói dữ liệu **không có** `video_1/eval_config.json` mà `evaluate_practice.py` cần. Ô dưới tạo file đó với
benchmark trung tính `LAB` — đây chỉ là tên thư mục trong TrackEval. Gói cũng không có `preview/`; video có vẽ ID
do `run_tracking.py --save-video` sinh ra là đủ để xem.

In [ ]:
import json

DATA_ID = "1UeVPQd6j5pSzxoJDcKJrerT9SL3vJLDt"
ZIP = Path("/content/data_lab21.zip")
LAB_ROOT = Path("/content/lab_data")


def tim_lab_data(root: Path):
    # Trả về thư mục chứa video_1 … video_5 sau khi giải nén, hoặc None.
    for p in sorted(root.rglob("video_1")):
        if "__MACOSX" not in p.parts and (p / "img1").is_dir():
            return p.parent
    return None


LAB_DATA = tim_lab_data(LAB_ROOT) if LAB_ROOT.exists() else None
if LAB_DATA is None:
    if not ZIP.exists():
        !gdown -q {DATA_ID} -O {ZIP}
    LAB_ROOT.mkdir(parents=True, exist_ok=True)
    !unzip -q -o {ZIP} -d {LAB_ROOT} -x "__MACOSX/*" "*.DS_Store"
    LAB_DATA = tim_lab_data(LAB_ROOT)
os.environ["LAB_DATA"] = str(LAB_DATA)
print("LAB_DATA =", LAB_DATA)

EVAL_CFG = LAB_DATA / "video_1" / "eval_config.json"
if not EVAL_CFG.exists():
    EVAL_CFG.write_text(json.dumps({"benchmark": "LAB", "split": "train"}))
    print("Đã tạo", EVAL_CFG)
!python scripts/check_data.py --lab-data-root {LAB_DATA}

## 3. Hàm tiện ích

- `chay(video, tracker, conf, iou, max_frames=...)` gọi `scripts/run_tracking.py`. Mỗi cấu hình ghi vào thư mục riêng
  `runs/thu/<video>/<tracker>_c<conf>_i<iou>[_f<max_frames>]/`, chạy lại cùng cấu hình thì dùng lại kết quả cũ.
- `cham(txt, run_name)` gọi `scripts/evaluate_practice.py` (chỉ `video_1`), trả về HOTA / MOTA / IDF1.
- `thong_ke(txt)` đếm số ID, số ID sống < 10 frame, số hộp mỗi frame. **Không phải metric** — chỉ gợi ý chỗ cần xem kỹ.
  Nhiều ID ngắn thường là hộp giả hoặc track bị đứt rồi mở ID mới.
- `xem(txt1, txt2, ...)` hiện các video có vẽ ID cạnh nhau.

In [ ]:
import base64

import pandas as pd
from IPython.display import HTML, display

sys.path.insert(0, str(REPO / "scripts"))
from check_submission import summarize_mot_text
from evaluate_practice import parse_summary, summary_path

BENCHMARK = json.loads(EVAL_CFG.read_text())["benchmark"]
VIDEOS = ["video_1", "video_2", "video_3", "video_4", "video_5"]
MOTION = ["bytetrack", "ocsort"]  # chủ yếu dựa vào chuyển động
REID = ["botsort", "strongsort", "deepocsort"]  # thêm ngoại hình (Re-ID)


def chay(video, tracker, conf=0.3, iou=0.5, out=None, max_frames=0, save_video=True, lam_lai=False):
    # Chạy scripts/run_tracking.py cho một cấu hình, trả về đường dẫn video_N.txt.
    if out is None:
        out = f"runs/thu/{video}/{tracker}_c{conf}_i{iou}" + (f"_f{max_frames}" if max_frames else "")
    out = Path(out)
    txt = out / f"{video}.txt"
    preview = out / f"{video}_preview.mp4"
    if txt.exists() and (preview.exists() or not save_video) and not lam_lai:
        print(f"[{video}] {tracker} conf={conf} iou={iou}: đã có kết quả, dùng lại ({out})")
        return txt
    cmd = [
        sys.executable, "scripts/run_tracking.py",
        "--source", str(LAB_DATA / video / "img1"),
        "--seq-name", video,
        "--tracker", tracker, "--conf", str(conf), "--iou", str(iou),
        "--out", str(out), "--device", DEVICE,
    ]
    if save_video:
        cmd.append("--save-video")
    if max_frames:
        cmd += ["--max-frames", str(max_frames)]
    r = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace")
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError(f"run_tracking.py lỗi: {video} {tracker}")
    dong = [line for line in r.stdout.splitlines() if line.startswith(f"[{video}]")]
    print(dong[-1] if dong else r.stdout[-500:])
    return txt


def cham(txt, run_name):
    # Chấm video_1.txt, trả về (dict số liệu, toàn bộ output của TrackEval).
    r = subprocess.run(
        [sys.executable, "scripts/evaluate_practice.py",
         "--trackeval-root", "TrackEval", "--lab-data-root", str(LAB_DATA),
         "--submission", str(txt), "--run-name", run_name],
        capture_output=True, text=True, encoding="utf-8", errors="replace",
    )
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError(f"evaluate_practice.py lỗi: {run_name}")
    s = parse_summary(summary_path(Path("TrackEval"), run_name, BENCHMARK, "train").read_text())
    return {k: s[k] for k in ("HOTA", "DetA", "AssA", "MOTA", "IDF1", "IDSW")}, r.stdout


def thong_ke(txt):
    # Số liệu thô của một file kết quả — gợi ý để xem video, không thay được nhãn.
    s = summarize_mot_text(Path(txt).read_text())
    return {
        "ID": s["n_ids"],
        "ID ngắn (<10 frame)": s["n_short_ids"],
        "hộp/frame": round(s["n_rows"] / max(s["max_frame"], 1), 1),
        "frame cuối": s["max_frame"],
    }


def _h264(mp4, width):
    # Trình duyệt không phát được mp4v của OpenCV -> chuyển sang H.264, thu nhỏ cho nhẹ.
    out = mp4.with_name(f"{mp4.stem}_h264_{width}.mp4")
    if not out.exists():
        subprocess.run(
            ["ffmpeg", "-y", "-loglevel", "error", "-i", str(mp4), "-vf", f"scale={width}:-2",
             "-c:v", "libx264", "-preset", "veryfast", "-crf", "30", "-pix_fmt", "yuv420p", str(out)],
            check=True,
        )
    return out


def xem(*txts, nhan=None, width=640):
    # Hiện video xem thử (cùng thư mục với video_N.txt) cạnh nhau.
    o = []
    for i, txt in enumerate(txts):
        txt = Path(txt)
        mp4 = txt.with_name(f"{txt.stem}_preview.mp4")
        if not mp4.exists():
            print("Thiếu", mp4, "— chạy lại với save_video=True")
            continue
        data = base64.b64encode(_h264(mp4, width).read_bytes()).decode()
        title = nhan[i] if nhan else txt.parent.name
        o.append(
            f'<div style="flex:1;min-width:320px"><b>{title}</b><br>'
            f'<video width="100%" controls loop muted src="data:video/mp4;base64,{data}"></video></div>'
        )
    display(HTML('<div style="display:flex;gap:8px;flex-wrap:wrap">' + "".join(o) + "</div>"))


def so_sanh(video, trackers, conf=0.3, iou=0.5, max_frames=300):
    # Chạy nhiều tracker cùng ngưỡng, in bảng thống kê thô và hiện video cạnh nhau.
    txts = [chay(video, t, conf, iou, max_frames=max_frames) for t in trackers]
    display(pd.DataFrame([{"tracker": t, **thong_ke(x)} for t, x in zip(trackers, txts)]))
    xem(*txts, nhan=[f"{t} conf={conf} iou={iou}" for t in trackers])


def quet(video, tracker, conf=0.3, iou=0.5, max_frames=300):
    # Quét MỘT tham số: truyền tuple cho conf HOẶC iou, tham số còn lại giữ cố định.
    confs = conf if isinstance(conf, (tuple, list)) else (conf,)
    ious = iou if isinstance(iou, (tuple, list)) else (iou,)
    if len(confs) > 1 and len(ious) > 1:
        raise ValueError("Mỗi lần chỉ đổi một tham số: conf hoặc iou")
    cau_hinh = [(c, i) for c in confs for i in ious]
    txts = [chay(video, tracker, c, i, max_frames=max_frames) for c, i in cau_hinh]
    display(pd.DataFrame([{"tracker": tracker, "conf": c, "iou": i, **thong_ke(x)} for (c, i), x in zip(cau_hinh, txts)]))
    xem(*txts, nhan=[f"{tracker} conf={c} iou={i}" for c, i in cau_hinh])

## 4. Baseline — `video_1` + ByteTrack, 150 frame

Mỗi người một màu. Một người đang đi đều mà **đổi màu** là tracker đã đổi ID.

In [ ]:
txt = chay("video_1", "bytetrack", 0.3, 0.5, out="runs/thu_nhanh", max_frames=150)
print(thong_ke(txt))
xem(txt)

## 5. `video_1` — có nhãn, so bằng số

### 5a. Năm tracker, cùng `conf=0.3`, `iou=0.5`, chạy đủ 600 frame

Chạy đủ frame vì nhãn phủ cả video — chạy thiếu frame sẽ bị tính là bỏ sót. Không lưu video để nhanh hơn.

In [ ]:
KQ_V1 = {}  # (tracker, conf, iou) -> dict số liệu


def thu_v1(tracker, conf, iou):
    # Chạy + chấm video_1 cho một cấu hình (dùng lại nếu đã chấm).
    key = (tracker, conf, iou)
    if key not in KQ_V1:
        txt = chay("video_1", tracker, conf, iou, save_video=False)
        KQ_V1[key], _ = cham(txt, f"{tracker}_c{conf}_i{iou}")
    return {"tracker": tracker, "conf": conf, "iou": iou, **KQ_V1[key]}


bang_tracker = pd.DataFrame([thu_v1(t, 0.3, 0.5) for t in MOTION + REID]).sort_values("HOTA", ascending=False)
display(bang_tracker.round(2))

### 5b. Quét ngưỡng cho tracker tốt nhất — mỗi lần đổi một số

Mặc định lấy tracker có HOTA cao nhất ở 5a. Muốn chọn khác thì sửa `TRACKER_V1`.

In [ ]:
TRACKER_V1 = bang_tracker.iloc[0]["tracker"]
print("Tracker:", TRACKER_V1)

bang_conf = pd.DataFrame([thu_v1(TRACKER_V1, c, 0.5) for c in (0.15, 0.3, 0.5)])
display(bang_conf.round(2))
CONF_V1 = float(bang_conf.sort_values("HOTA", ascending=False).iloc[0]["conf"])

bang_iou = pd.DataFrame([thu_v1(TRACKER_V1, CONF_V1, i) for i in (0.4, 0.5, 0.7)])
display(bang_iou.round(2))
IOU_V1 = float(bang_iou.sort_values("HOTA", ascending=False).iloc[0]["iou"])

print(f"\nCấu hình video_1 theo HOTA: {TRACKER_V1}, conf={CONF_V1}, iou={IOU_V1}")
bang_v1 = pd.DataFrame([{"tracker": k[0], "conf": k[1], "iou": k[2], **v} for k, v in KQ_V1.items()])
Path("runs").mkdir(exist_ok=True)
bang_v1.sort_values("HOTA", ascending=False).to_csv("runs/video_1_tat_ca_cau_hinh.csv", index=False)

### 5c. Xem bằng mắt: cấu hình tốt nhất so với baseline ByteTrack

Đối chiếu số với hình: IDSW cao thì tìm chỗ hai người đi cắt nhau; DetA thấp thì tìm người rõ mà không có hộp.

In [ ]:
xem(
    chay("video_1", "bytetrack", 0.3, 0.5, max_frames=300),
    chay("video_1", TRACKER_V1, CONF_V1, IOU_V1, max_frames=300),
    nhan=["baseline bytetrack 0.3/0.5", f"chọn: {TRACKER_V1} {CONF_V1}/{IOU_V1}"],
)

## 6. `video_2`–`video_5` — không nhãn, đánh giá bằng mắt

Với **mỗi** video:

1. `so_sanh(...)`: một tracker chuyển động và một tracker Re-ID, cùng ngưỡng, 300 frame đầu.
2. Chọn tracker có vẻ hơn, gán vào `TRACKER`, rồi `quet(...)`: đổi `conf` (giữ `iou`), sau đó đổi `iou` (giữ `conf`).
3. Ghi lại điều bạn thấy vào ô ghi chú — dùng cho cột *Quan sát* và *Đã thử nhưng loại* trong báo cáo.

Gợi ý khi xem:

- Cùng một người đi đều mà đổi màu ID → **đổi danh tính**.
- Hộp nhảy sang người bên cạnh khi hai người cắt nhau → **gán nhầm** (hay gặp ở cảnh đông, tracker chỉ dùng chuyển động).
- Người rõ trong ảnh nhưng không có hộp → `conf` có thể đang **cao**.
- Hộp nhấp nháy trên nền, bóng, vật không phải người → `conf` có thể đang **thấp**.
- Camera di chuyển / rung → để ý ID có bị mất khi khung hình giật không (BoT-SORT có bù chuyển động camera).

### `video_2` — Phố, camera tĩnh trên cao, **ban đêm**, **rất đông**

Ảnh tối làm đặc trưng ngoại hình kém tin cậy; người nhỏ và sát nhau. Re-ID có giúp hay ngược lại? `conf` thấp có cứu được người tối không, hay sinh hộp giả?

In [ ]:
so_sanh("video_2", ["bytetrack", "botsort"])  # có thể đổi: ocsort / strongsort / deepocsort

In [ ]:
TRACKER = "botsort"  # <- tracker có vẻ hơn ở ô trên
quet("video_2", TRACKER, conf=(0.15, 0.3, 0.5), iou=0.5)

In [ ]:
CONF = 0.3  # <- conf có vẻ hợp nhất ở ô trên
quet("video_2", TRACKER, conf=CONF, iou=(0.4, 0.5, 0.7))

**Ghi chú `video_2`** (bấm đúp để sửa):

- Tracker chuyển động vs Re-ID: …
- `conf`: …
- `iou`: …
- Chọn: … — Đã thử nhưng loại: …

### `video_3` — Camera **di chuyển**, ảnh nhỏ, **ít khung hình/giây**

FPS thấp nghĩa là người dịch chuyển xa giữa hai frame — dự đoán chuyển động dễ trượt. Tracker nào giữ ID qua bước nhảy lớn?

In [ ]:
so_sanh("video_3", ["bytetrack", "botsort"])  # có thể đổi: ocsort / strongsort / deepocsort

In [ ]:
TRACKER = "botsort"  # <- tracker có vẻ hơn ở ô trên
quet("video_3", TRACKER, conf=(0.15, 0.3, 0.5), iou=0.5)

In [ ]:
CONF = 0.3  # <- conf có vẻ hợp nhất ở ô trên
quet("video_3", TRACKER, conf=CONF, iou=(0.4, 0.5, 0.7))

**Ghi chú `video_3`** (bấm đúp để sửa):

- Tracker chuyển động vs Re-ID: …
- `conf`: …
- `iou`: …
- Chọn: … — Đã thử nhưng loại: …

### `video_4` — **Trong nhà**, camera tiến tới, **kính phản chiếu**

Bóng phản chiếu trong kính có bị nhận là người không (`conf`)? Người to dần khi camera tiến tới — ID có giữ được không?

In [ ]:
so_sanh("video_4", ["bytetrack", "botsort"])  # có thể đổi: ocsort / strongsort / deepocsort

In [ ]:
TRACKER = "botsort"  # <- tracker có vẻ hơn ở ô trên
quet("video_4", TRACKER, conf=(0.15, 0.3, 0.5), iou=0.5)

In [ ]:
CONF = 0.3  # <- conf có vẻ hợp nhất ở ô trên
quet("video_4", TRACKER, conf=CONF, iou=(0.4, 0.5, 0.7))

**Ghi chú `video_4`** (bấm đúp để sửa):

- Tracker chuyển động vs Re-ID: …
- `conf`: …
- `iou`: …
- Chọn: … — Đã thử nhưng loại: …

### `video_5` — Trên **xe bus**, giao lộ đông, **rung lắc**

Rung lắc làm cả khung hình xê dịch. So tracker có bù chuyển động camera với tracker chỉ dựa vào chuyển động.

In [ ]:
so_sanh("video_5", ["bytetrack", "botsort"])  # có thể đổi: ocsort / strongsort / deepocsort

In [ ]:
TRACKER = "botsort"  # <- tracker có vẻ hơn ở ô trên
quet("video_5", TRACKER, conf=(0.15, 0.3, 0.5), iou=0.5)

In [ ]:
CONF = 0.3  # <- conf có vẻ hợp nhất ở ô trên
quet("video_5", TRACKER, conf=CONF, iou=(0.4, 0.5, 0.7))

**Ghi chú `video_5`** (bấm đúp để sửa):

- Tracker chuyển động vs Re-ID: …
- `conf`: …
- `iou`: …
- Chọn: … — Đã thử nhưng loại: …

## 7. Chạy bản nộp — đủ frame, cùng thư mục `runs/nop_bai/`

Điền cấu hình **bạn đã chọn** ở phần 5–6. `video_1` lấy sẵn từ bảng số phần 5b.
Các giá trị đang ghi cho `video_2`–`video_5` là cấu hình nhóm đã nộp (lý do trong `BAO_CAO_mau.md`). Đổi nếu bạn thấy khác trên video.

In [ ]:
CAU_HINH = {
    # video: (tracker, conf, iou)
    "video_1": (TRACKER_V1, CONF_V1, IOU_V1),
    "video_2": ("botsort", 0.15, 0.5),
    "video_3": ("ocsort", 0.3, 0.5),
    "video_4": ("deepocsort", 0.3, 0.5),
    "video_5": ("botsort", 0.15, 0.5),
}

for video, (tracker, conf, iou) in CAU_HINH.items():
    assert tracker in MOTION + REID, f"{video}: tracker không hợp lệ {tracker}"
    chay(video, tracker, conf, iou, out="runs/nop_bai", lam_lai=True)

pd.DataFrame([{"video": v, "tracker": t, "conf": c, "iou": i} for v, (t, c, i) in CAU_HINH.items()]).to_csv(
    "runs/nop_bai/cau_hinh.csv", index=False
)
!python scripts/check_submission.py --lab-data-root {LAB_DATA} --submission-dir runs/nop_bai

Xem lại cả 5 video bản nộp (đủ frame) trước khi viết quan sát:

In [ ]:
for video in VIDEOS:
    tracker, conf, iou = CAU_HINH[video]
    xem(f"runs/nop_bai/{video}.txt", nhan=[f"{video}: {tracker} conf={conf} iou={iou}"], width=800)

## 8. Chấm `video_1` bản nộp

Dán bảng in ra dưới đây vào mục **2. Số liệu video_1** của `submission_template/BAO_CAO_mau.md`.
Đổi `RUN_NAME` thành tên nhóm bạn.

In [ ]:
RUN_NAME = "nhom01_video1"
so, output = cham("runs/nop_bai/video_1.txt", RUN_NAME)
print(output.split("Timing analysis")[0])
print({k: round(v, 2) for k, v in so.items()})

## 9. Lưu kết quả

Tải `nop_bai.zip` về máy (gồm `video_1.txt` … `video_5.txt`, video có vẽ ID, `cau_hinh.csv`) và bảng
`video_1_tat_ca_cau_hinh.csv`. Ô thứ hai lưu thêm vào Google Drive nếu bạn muốn.

Khi nộp: chép 5 file `.txt` vào `runs/nop_bai/` trong repo, điền báo cáo. **Không** commit video và `lab_data/`.

In [ ]:
!cd runs && zip -qr /content/nop_bai.zip nop_bai video_1_tat_ca_cau_hinh.csv -x "*_h264_*"
!ls -lh /content/nop_bai.zip
from google.colab import files
files.download("/content/nop_bai.zip")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/lab_tracking && cp /content/nop_bai.zip /content/drive/MyDrive/lab_tracking/
!ls -lh /content/drive/MyDrive/lab_tracking